# Notebook 1 — Data Understanding & Cleaning

**Objective:** Load the raw Zomato restaurant extract, understand its structure, and produce a
cleaned dataset that is safe to build EDA and modeling on top of.

**Scope of this notebook:**
- Structural checks (duplicates, constants, missing values)
- Binary encoding of Yes/No fields
- **Currency normalization** — the single most important correction in this notebook. The raw
  `Average Cost for two` field is stored in each restaurant's local currency and is not
  comparable across countries as-is.
- Outlier review (documented, not blindly removed)
- Data validation assertions
- Target variable (`Aggregate rating`) understanding

See `docs/05_data_dictionary.md` and `docs/06_assumptions_and_limitations.md` for the full
rationale behind every decision made here.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "dataset" / "Res_dataset.csv"
OUTPUT_PATH = PROJECT_ROOT / "dataset" / "cleaned_dataset.csv"

df = pd.read_csv(DATA_PATH)
print("Dataset Loaded Successfully.")
print("Shape:", df.shape)

Dataset Loaded Successfully.
Shape: (9551, 21)


## Duplicate Check
A real analyst never assumes a source extract is duplicate-free — this is checked explicitly,
not assumed.

In [3]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Restaurant IDs:", df.duplicated(subset="Restaurant ID").sum())
df.drop_duplicates(subset="Restaurant ID", inplace=True)

Duplicate rows: 0
Duplicate Restaurant IDs: 0


**Observation:** No duplicates found — this source extract was already deduplicated upstream by whoever produced it. Documented rather than silently skipped.

In [4]:
print(df.isnull().sum()[df.isnull().sum() > 0])
df["Cuisines"] = df["Cuisines"].fillna("Unknown")

Cuisines    9
dtype: int64


**Observation:** Only `Cuisines` has missing values (9 rows, 0.09%). Filled as `"Unknown"` rather than dropped, since dropping 9 rows for one missing categorical field is unnecessary data loss.

In [5]:
constant_cols = [c for c in df.columns if df[c].nunique() == 1]
print("Constant columns:", constant_cols)
df.drop(columns=constant_cols, inplace=True)

Constant columns: ['Switch to order menu']


**Observation:** `Switch to order menu` is constant across all 9,551 rows (a legacy/deprecated Zomato feature flag). Zero predictive or descriptive value — dropped.

In [6]:
yes_no_cols = ["Has Table booking", "Has Online delivery", "Is delivering now"]
df[yes_no_cols] = df[yes_no_cols].replace({"Yes": 1, "No": 0}).astype(int)
print("Converted:", yes_no_cols)

Converted: ['Has Table booking', 'Has Online delivery', 'Is delivering now']


C:\Users\venut\AppData\Local\Temp\ipykernel_27316\1450795683.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[yes_no_cols] = df[yes_no_cols].replace({"Yes": 1, "No": 0}).astype(int)


## Currency Normalization — Critical Correction

**The gap found in the original version of this notebook:** `Average Cost for two` is stored in
each restaurant's local currency, with no conversion applied anywhere downstream (EDA, modeling,
or the dashboard). This silently made every cross-country cost comparison meaningless — e.g. the
raw data's overall max of 800,000 isn't a data error, it's simply Indonesian Rupiah at its normal
scale.

**A second issue found while fixing the first:** the `Currency` text column itself is unreliable.
Four different country codes (14 Australia, 37 Canada, 184 Singapore, 216 USA) all display
`"Dollar($)"` even though these are four different currencies (AUD, CAD, SGD, USD). Country Code
162 (Philippines) is mislabeled `"Botswana Pula(P)"` — a bug in the original Zomato API scrape,
not something introduced here.

**Fix:** map the *actual* currency from `Country Code` directly (not the ambiguous `Currency`
text field), then convert every cost value to a common USD baseline using static, documented,
approximate 2019 average FX rates. These are historical approximations for normalization
purposes only — this is a historical snapshot, so live FX rates would be inappropriate and
misleading here.

In [7]:
country_currency = {
    1: "INR", 14: "AUD", 30: "BRL", 37: "CAD", 94: "IDR", 148: "NZD",
    162: "PHP", 166: "QAR", 184: "SGD", 189: "ZAR", 191: "LKR",
    208: "TRY", 214: "AED", 215: "GBP", 216: "USD"
}
country_name = {
    1: "India", 14: "Australia", 30: "Brazil", 37: "Canada", 94: "Indonesia",
    148: "New Zealand", 162: "Philippines", 166: "Qatar", 184: "Singapore",
    189: "South Africa", 191: "Sri Lanka", 208: "Turkey", 214: "UAE",
    215: "United Kingdom", 216: "United States"
}
# Static, documented, approximate 2019 average units of local currency per 1 USD.
# For historical-snapshot normalization only -- not live FX rates.
usd_rate_2019 = {
    "INR": 70.0, "AUD": 1.39, "BRL": 3.9, "CAD": 1.32, "IDR": 14000.0,
    "NZD": 1.44, "PHP": 52.0, "QAR": 3.64, "SGD": 1.35, "ZAR": 14.4,
    "LKR": 178.0, "TRY": 5.7, "AED": 3.67, "GBP": 0.78, "USD": 1.0
}

df["Country Name"] = df["Country Code"].map(country_name)
df["Actual Currency"] = df["Country Code"].map(country_currency)
df["USD Conversion Rate"] = df["Actual Currency"].map(usd_rate_2019)
df["Average Cost for Two (USD)"] = (df["Average Cost for two"] / df["USD Conversion Rate"]).round(2)

print("Mean cost BEFORE normalization (raw local currency):")
print(df.groupby("Country Name")["Average Cost for two"].mean().round(2))
print()
print("Mean cost AFTER normalization (USD):")
print(df.groupby("Country Name")["Average Cost for Two (USD)"].mean().round(2))

Mean cost BEFORE normalization (raw local currency):
Country Name
Australia             24.08
Brazil               134.67
Canada                36.25
India                623.37
Indonesia         281190.48
New Zealand           69.75
Philippines         1606.82
Qatar                223.75
Singapore            155.75
South Africa         419.73
Sri Lanka           2375.00
Turkey                84.85
UAE                  166.42
United Kingdom        47.81
United States         26.15
Name: Average Cost for two, dtype: float64

Mean cost AFTER normalization (USD):
Country Name
Australia          17.33
Brazil             34.53
Canada             27.46
India               8.91
Indonesia          20.09
New Zealand        48.44
Philippines        30.90
Qatar              61.47
Singapore         115.37
South Africa       29.15
Sri Lanka          13.34
Turkey             14.89
UAE                45.35
United Kingdom     61.30
United States      26.15
Name: Average Cost for Two (USD), dtype: floa

**Observation:** Before normalization, Indonesia appears ~450x more expensive than India on
paper (₹281,190 vs ₹623) purely because of currency denomination — not real price positioning.
After conversion, Indonesia (~$20) and India (~$9) sit in a sane, comparable range, and Singapore
correctly emerges as the most expensive market (~$115) rather than being masked by raw-currency
scale. **Every cost-based statistic from this point forward in the project uses
`Average Cost for Two (USD)`, never the raw local-currency field.**

In [8]:
print(df["Average Cost for Two (USD)"].describe())
q1 = df["Average Cost for Two (USD)"].quantile(0.25)
q3 = df["Average Cost for Two (USD)"].quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
n_outliers = (df["Average Cost for Two (USD)"] > upper).sum()
print(f"\nIQR upper bound: {upper:.2f} USD | Restaurants above bound: {n_outliers} ({n_outliers/len(df)*100:.2f}%)")

count    9551.000000
mean       11.278201
std        14.926171
min         0.000000
25%         4.290000
50%         7.140000
75%        11.430000
max       370.370000
Name: Average Cost for Two (USD), dtype: float64

IQR upper bound: 22.14 USD | Restaurants above bound: 1123 (11.76%)


**Observation:** 11.76% of restaurants sit above the IQR outlier bound even after currency normalization. These are treated as genuine premium restaurants, not data errors — retained, and addressed via a log transform (`Log_Average_Cost_USD`) at the feature-engineering stage rather than removed outright.

In [9]:
print(df["Votes"].describe())
v_q1, v_q3 = df["Votes"].quantile([0.25, 0.75])
v_upper = v_q3 + 1.5 * (v_q3 - v_q1)
v_out = (df["Votes"] > v_upper).sum()
print(f"\nIQR upper bound: {v_upper:.1f} | Restaurants above bound: {v_out} ({v_out/len(df)*100:.2f}%)")

count     9551.000000
mean       156.909748
std        430.169145
min          0.000000
25%          5.000000
50%         31.000000
75%        131.000000
max      10934.000000
Name: Votes, dtype: float64

IQR upper bound: 320.0 | Restaurants above bound: 1126 (11.79%)


**Decision:** High-vote restaurants (11.79% above bound) are genuinely popular restaurants, not data errors — retained. `Votes` will be excluded entirely from the predictive model (see BRD BR-3 and Notebook 3), so its distribution is documented here for completeness but does not affect the pre-launch model.

## Data Validation Assertions
A cleaned dataset should prove its own sanity, not just assume it.

In [10]:
assert df["Aggregate rating"].between(0, 5).all(), "Rating out of [0,5] range"
assert df["Price range"].isin([1, 2, 3, 4]).all(), "Price range outside {1,2,3,4}"
assert df["Latitude"].between(-90, 90).all(), "Latitude out of range"
assert df["Longitude"].between(-180, 180).all(), "Longitude out of range"
print("All validation checks passed: ratings in [0,5], price range in {1,2,3,4}, "
      "lat/long within valid geographic bounds.")

All validation checks passed: ratings in [0,5], price range in {1,2,3,4}, lat/long within valid geographic bounds.


In [11]:
print("Min:", df["Aggregate rating"].min())
print("Max:", df["Aggregate rating"].max())
print("Mean (all rows):", round(df["Aggregate rating"].mean(), 3))
zero_count = (df["Aggregate rating"] == 0).sum()
print(f"Restaurants with 0 rating: {zero_count} ({zero_count/len(df)*100:.2f}%)")
rated = df[df["Aggregate rating"] > 0]
print("Mean rating (excluding 0s):", round(rated["Aggregate rating"].mean(), 3))
print("Median rating (excluding 0s):", round(rated["Aggregate rating"].median(), 3))

Min: 0.0
Max: 4.9
Mean (all rows): 2.666
Restaurants with 0 rating: 2148 (22.49%)
Mean rating (excluding 0s): 3.44
Median rating (excluding 0s): 3.4


**Observation:** 22.49% of restaurants have a `0.0` rating. This is not treated as a genuine
low rating — it almost certainly represents "not yet reviewed" rather than "reviewed and rated
0." These rows will be excluded from model training in Notebook 3 (a judgment call, documented
in `docs/06_assumptions_and_limitations.md` rather than made silently).

In [12]:
country_counts = df["Country Name"].value_counts()
print(country_counts)
india_share = country_counts["India"] / len(df) * 100
print(f"\nIndia share: {india_share:.2f}%")
small_countries = country_counts[country_counts < 40]
print(f"Countries with <40 restaurants (descriptive-only, per BRD): {small_countries.index.tolist()}")

Country Name
India             8652
United States      434
United Kingdom      80
Brazil              60
South Africa        60
UAE                 60
New Zealand         40
Turkey              34
Australia           24
Philippines         22
Indonesia           21
Qatar               20
Singapore           20
Sri Lanka           20
Canada               4
Name: count, dtype: int64

India share: 90.59%
Countries with <40 restaurants (descriptive-only, per BRD): ['Turkey', 'Australia', 'Philippines', 'Indonesia', 'Qatar', 'Singapore', 'Sri Lanka', 'Canada']


**Observation:** India represents 90.59% of the dataset. 8 of 15 countries have fewer than 40 restaurants. Per `docs/02_business_requirements.md` (BR), any per-country statistic for these 8 countries is reported descriptively only and never used to justify a business recommendation — this is enforced consistently in Notebooks 2 and 4.

In [13]:
df["Country Code"] = df["Country Code"].astype("category")
df["Price range"] = df["Price range"].astype("category")

df.to_csv(OUTPUT_PATH, index=False)
print("Saved cleaned_dataset.csv. Final shape:", df.shape)

Saved cleaned_dataset.csv. Final shape: (9551, 24)


## Notebook 1 Summary

- Deduplicated on `Restaurant ID` (0 found), dropped 1 constant column, filled 9 missing
  `Cuisines` values.
- **Fixed a real currency normalization gap** — costs are now comparable in USD across all 15
  countries.
- **Caught and documented a source-data bug** — `Country Code` 162 was mislabeled with the wrong
  currency in the raw extract.
- Ran outlier review (documented, not blindly stripped) and 4 data validation assertions.
- Confirmed the 90.59% India skew and the 22.49% zero-rating rate — both carried forward as
  explicit constraints into every later notebook.

**Output:** `Dataset/cleaned_dataset.csv` (9,551 rows × 24 columns)